# Jupyter Notebook: Tiền xử lý & Phân tích Chuyên sâu (DUONG)

- **Người thực hiện:** Châu Thanh Dương
- **Ngành hàng:** Gia Dụng (Kitchen & Home Appliances)
- **Phương pháp:** Phân tích giá tâm lý & Chỉ số uy tín gian hàng

**Mục tiêu SMART:**
1. **Phân tích hiệu ứng số "9":** So sánh hiệu quả lượt bán giữa giá tâm lý (đuôi 9) và giá tròn.
2. **Chỉ số niềm tin (Trust Index):** Đánh giá sự tương quan giữa khả năng tương tác (Rating/Review) và doanh số thực tế.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Cấu hình hiển thị số thập phân
pd.options.display.float_format = '{:,.2f}'.format

# Đọc dữ liệu thô
fact_df = pd.read_csv(r'C:\Users\ctduo\OneDrive\Documents\DV\LAB 01\Lab_01_Data_Vizualization\data\raw\fact_product_tiki_20260330_duong.csv')
shop_df = pd.read_csv(r'C:\Users\ctduo\OneDrive\Documents\DV\LAB 01\Lab_01_Data_Vizualization\data\raw\dim_shop_tiki_20260330_duong.csv')

# Merge dữ liệu: Lấy thêm thông tin Shop vào bảng Sản phẩm
df = pd.merge(fact_df, shop_df[['shop_id', 'shop_name', 'is_mall']], on='shop_id', how='left')

print(f"📊 Quy mô dữ liệu: {df.shape[0]} sản phẩm | {df.shape[1]} thuộc tính")
df.head(3)

In [ ]:
# 1. Ép kiểu dữ liệu (Sử dụng dictionary để xử lý hàng loạt)
type_map = {
    'price_current': float,
    'price_original': float,
    'sold_count': float,
    'rating': float,
    'review_count': int
}
for col, t in type_map.items():
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(t)

# 2. Xử lý logic giá (Outliers & Missing logic)
# Nếu giá gốc thấp hơn giá hiện tại (lỗi dữ liệu), gán giá gốc = giá hiện tại
df['price_original'] = np.where(df['price_original'] < df['price_current'], df['price_current'], df['price_original'])

# 3. Lọc bỏ Outliers: Chỉ giữ sản phẩm từ 10k đến 50 triệu (phù hợp ngành Gia dụng)
df = df[(df['price_current'] >= 10000) & (df['price_current'] <= 50000000)].copy()

# 4. Ép kiểu Boolean cho các nhãn
for col in ['is_mall', 'is_freeship', 'has_video']:
    df[col] = df[col].astype(bool)

print(" Hoàn tất làm sạch. Dữ liệu sẵn sàng phân tích.")

In [ ]:
# --- SMART 1: Hiệu ứng giá tâm lý (Số 9) ---
# Kiểm tra giá có kết thúc bằng cụm '900', '90' hoặc '9' (đặc thù VNĐ)
def is_psychological_price(p):
    s = str(int(p))
    return "9" in s[-3:] # Kiểm tra 3 số cuối có chứa số 9 không

df['is_psy_price'] = df['price_current'].apply(is_psychological_price)

# --- SMART 2: Chỉ số niềm tin (Trust Index) ---
# Công thức: (Rating * 0.7) + (Log của lượt Review * 0.3)
# Thêm 1 vào review_count để tránh lỗi log(0)
df['trust_index_score'] = (df['rating'] * 0.7) + (np.log1p(df['review_count']) * 0.3)

# Cột bổ sung: Doanh thu ước tính
df['revenue_est'] = df['price_current'] * df['sold_count']

print(" Đã tạo thành công các cột derived: is_psy_price, trust_index_score, revenue_est")

In [5]:
# So sánh lượt bán trung bình giữa 2 nhóm giá
psy_viz = df.groupby('is_psy_price')['sold_count'].mean().reset_index()
psy_viz['Label'] = psy_viz['is_psy_price'].map({True: 'Giá đuôi 9 (Tâm lý)', False: 'Giá thông thường'})

fig1 = px.bar(
    psy_df := psy_viz.sort_values('sold_count', ascending=False),
    x='Label', 
    y='sold_count',
    color='Label',
    text_auto='.1f',
    title="[DUONG] - So sánh Lượt bán Trung bình: Hiệu ứng giá đuôi 9",
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig1.update_layout(showlegend=False, yaxis_title="Lượt bán trung bình")
fig1.show()

In [ ]:
# Vẽ biểu đồ phân tán để thấy sự tương quan
fig2 = px.scatter(
    df[df['sold_count'] > 0], # Chỉ xem các sp đã có lượt bán
    x='trust_index_score',
    y='sold_count',
    size='revenue_est',
    color='is_mall',
    hover_name='product_name',
    title="[DUONG] - Tương quan giữa Chỉ số niềm tin và Hiệu quả bán hàng",
    labels={'trust_index_score': 'Trust Index (Rating & Review)', 'sold_count': 'Lượt bán'}
)
fig2.show()